In [1]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
import torch
from src.surgical_copilot.HemoDataset import HemoDataSet
from src.surgical_copilot.bench.perturbation import PerturbationPipelines


plt.rcParams["figure.figsize"] = (18, 8)

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
ROOT_DIR = "data/raw"
PATIENT_ID = "pig1"
INDEX = 0

In [ ]:
dataset = HemosetDataSet(
    root_dir=ROOT_DIR,
    image_size=(480, 640),  # MONAI = (H, W)
    seed=42
)

sample = dataset.get_sample(
    split="train",
    patient_id=PATIENT_ID,
    index=INDEX,
    transform=False
)

sample_tf = dataset.get_sample(
    split="train",
    patient_id=PATIENT_ID,
    index=INDEX,
    transform=True
)

In [ ]:
raw_img_path = sample["image"]
raw_gt_path = sample["label"]

raw_img = np.array(Image.open(raw_img_path).convert("RGB"))
raw_gt = np.array(Image.open(raw_gt_path))

print("=" * 50)
print("RAW IMAGE")
print("=" * 50)

print("Path:", raw_img_path)
print("Shape:", raw_img.shape)
print("Height:", raw_img.shape[0])
print("Width:", raw_img.shape[1])

print("\nPIL size (W,H):", Image.open(raw_img_path).size)

print("\nGT shape:", raw_gt.shape)
print("GT unique values:", np.unique(raw_gt))
print("GT foreground ratio:", (raw_gt > 0).mean())

In [ ]:
raw_img_path = sample["image"]
raw_gt_path = sample["label"]

raw_img = np.array(Image.open(raw_img_path).convert("RGB"))
raw_gt = np.array(Image.open(raw_gt_path))

print("=" * 50)
print("RAW IMAGE")
print("=" * 50)

print("Path:", raw_img_path)
print("Shape:", raw_img.shape)
print("Height:", raw_img.shape[0])
print("Width:", raw_img.shape[1])

print("\nPIL size (W,H):", Image.open(raw_img_path).size)

print("\nGT shape:", raw_gt.shape)
print("GT unique values:", np.unique(raw_gt))
print("GT foreground ratio:", (raw_gt > 0).mean())

In [ ]:
img_tf = sample_tf["image"].cpu().numpy()
gt_tf = sample_tf["label"].cpu().numpy()

print("=" * 50)
print("TRANSFORMED")
print("=" * 50)

print("Image shape:", img_tf.shape)
print("GT shape:", gt_tf.shape)

print("Image min:", img_tf.min())
print("Image max:", img_tf.max())
print("Image mean:", img_tf.mean())
print("Image std:", img_tf.std())

print("GT unique:", np.unique(gt_tf))

In [ ]:
img_tf_vis = np.transpose(img_tf, (1, 2, 0))
gt_tf_vis = gt_tf.squeeze()

fig, ax = plt.subplots(2, 2, figsize=(16, 10))

ax[0,0].imshow(raw_img)
ax[0,0].set_title("Raw Image")

ax[0,1].imshow(raw_gt, cmap="gray")
ax[0,1].set_title("Raw GT")

ax[1,0].imshow(img_tf_vis)
ax[1,0].set_title("After Base Transforms")

ax[1,1].imshow(gt_tf_vis, cmap="gray")
ax[1,1].set_title("GT After Transform")

for a in ax.ravel():
    a.axis("off")

plt.show()

In [ ]:
overlay = raw_img.copy()

mask = raw_gt > 0

overlay[mask] = (
    0.6 * overlay[mask] +
    0.4 * np.array([255, 0, 0])
).astype(np.uint8)

plt.figure(figsize=(10,8))
plt.imshow(overlay)
plt.title("Ground Truth Overlay")
plt.axis("off")
plt.show()

In [ ]:
scenarios = PerturbationPipelines.get_eval_scenarios()

base_dict = {
    "image": sample_tf["image"].clone(),
    "label": sample_tf["label"].clone()
}

In [ ]:
n = len(scenarios)

fig, axes = plt.subplots(2, 3, figsize=(22, 12))
axes = axes.flatten()

for idx, (name, transform) in enumerate(scenarios.items()):

    data = {
        "image": base_dict["image"].clone(),
        "label": base_dict["label"].clone()
    }

    out = transform(data)

    img = out["image"]

    if torch.is_tensor(img):
        img = img.cpu().numpy()

    img = np.transpose(img, (1,2,0))
    img = np.clip(img, 0, 1)

    axes[idx].imshow(img)
    axes[idx].set_title(name)

    axes[idx].axis("off")

    print("="*50)
    print(name)
    print("min:", img.min())
    print("max:", img.max())
    print("mean:", img.mean())
    print("std:", img.std())

plt.tight_layout()
plt.show()

In [ ]:
worst_case = scenarios["chirurgical_worst_case"]

out = worst_case({
    "image": sample_tf["image"].clone(),
    "label": sample_tf["label"].clone()
})

worst_img = out["image"].cpu().numpy()
worst_img = np.transpose(worst_img, (1,2,0))
worst_img = np.clip(worst_img, 0, 1)

fig, ax = plt.subplots(1,2, figsize=(18,8))

ax[0].imshow(img_tf_vis)
ax[0].set_title("Clean")

ax[1].imshow(worst_img)
ax[1].set_title("Worst Case")

for a in ax:
    a.axis("off")

plt.show()